In [1]:
import boto3
print(f"Boto3 version: {boto3.__version__}")


Boto3 version: 1.43.89


In [24]:
usage_history = []

In [25]:
import boto3

region = "ap-south-1"
target_model_id = "anthropic.claude-haiku-4-5-20251001-v1:0"

session = boto3.Session(region_name=region)
creds = session.get_credentials()
print(f"credentials available: {creds is not None}")

bedrock = session.client("bedrock")
profiles = bedrock.list_inference_profiles(
    typeEquals="SYSTEM_DEFINED"
)["inferenceProfileSummaries"]

matching_profiles = [
    profile
    for profile in profiles
    if any(
        target_model_id in model.get("modelArn", "")
        for model in profile.get("models", [])
    )
]

if not matching_profiles:
    raise RuntimeError(
        f"No inference profile for {target_model_id} is available in {region}."
    )

model_id = matching_profiles[0]["inferenceProfileId"]
print(f"Using inference profile: {model_id}")

client = session.client("bedrock-runtime")
system_message = [
    {
        "text": (
            "You are a romantic poet. "
            "Write exactly six lines with warm, sincere language."
        )
    }
]
user_message = {
    "role": "user",
    "content": [{"text": "Write a 6 lines of poem about my besutiful lover and about the time we spend talking when she goes for a walk amidst woods in from her new albany phio home a\
                 and we speak on phone and we talk about our love and how we are going to spend our lives together"}],
}

response = client.converse(
    modelId=model_id,
    system = system_message,
    messages=[user_message],
)
print(response["output"]["message"]["content"][0]["text"])

credentials available: True
Using inference profile: global.anthropic.claude-haiku-4-5-20251001-v1:0
# Whispered Hearts Through the Pines

When autumn leaves beneath her feet do fall,
She walks through New Albany's ancient wood,
And through the phone, my darling hears my call,
Our voices weaving dreams as lovers should.
We speak of futures bright, of hands held fast,
A lifetime's love together, unsurpassed.


In [28]:



usage = response['usage']
print(f"the token usage is: {response['usage']}")
input_price_per_thousand = 0.0008   # Replace with current input-token price
output_price_per_thousand = 0.004 # Replace with current output-token price

estimated_cost = (
    usage["inputTokens"] / 1000 * input_price_per_thousand*95
    + usage["outputTokens"] / 1000 * output_price_per_thousand*95
)

usage_history.append({
    "input_tokens": usage.get("inputTokens", 0),
    "output_tokens": usage.get("outputTokens", 0),
    "total_tokens": usage.get("totalTokens", 0),
})


# print(f"Estimated cost in dollar: ${estimated_cost:.8f}")
print(f"Estimated cost in INR: {estimated_cost:.8f}")

the token usage is: {'inputTokens': 88, 'outputTokens': 81, 'totalTokens': 169, 'cacheReadInputTokens': 0}
Estimated cost in INR: 0.03746800


In [29]:
print(f"Usage history: {usage_history}")

response

Usage history: [{'input_tokens': 88, 'output_tokens': 81, 'total_tokens': 169}, {'input_tokens': 88, 'output_tokens': 81, 'total_tokens': 169}]


{'ResponseMetadata': {'RequestId': 'e2ba19bf-8f79-454d-8162-71d9821f226d',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 06 Sep 2026 12:30:09 GMT',
   'content-type': 'application/json',
   'content-length': '574',
   'connection': 'keep-alive',
   'x-amzn-requestid': 'e2ba19bf-8f79-454d-8162-71d9821f226d'},
  'RetryAttempts': 0},
 'output': {'message': {'role': 'assistant',
   'content': [{'text': "# Whispered Hearts Through the Pines\n\nWhen autumn leaves beneath her feet do fall,\nShe walks through New Albany's ancient wood,\nAnd through the phone, my darling hears my call,\nOur voices weaving dreams as lovers should.\nWe speak of futures bright, of hands held fast,\nA lifetime's love together, unsurpassed."}]}},
 'stopReason': 'end_turn',
 'usage': {'inputTokens': 88,
  'outputTokens': 81,
  'totalTokens': 169,
  'cacheReadInputTokens': 0},
 'metrics': {'latencyMs': 1656}}

In [35]:
response['output']['message']['content'][0]['text']

"# Whispered Hearts Through the Pines\n\nWhen autumn leaves beneath her feet do fall,\nShe walks through New Albany's ancient wood,\nAnd through the phone, my darling hears my call,\nOur voices weaving dreams as lovers should.\nWe speak of futures bright, of hands held fast,\nA lifetime's love together, unsurpassed."